# Controle de qualidade e trimming

Nesta prática:

1. FastQC nos reads brutos;
2. MultiQC para integrar relatórios;
3. Trimmomatic para processar os pares;
4. FastQC/MultiQC novamente;
5. comparação antes/depois.

Os parâmetros são **didáticos**. O artigo de *Hypochilus* utilizou
illumiprocessor/Trimmomatic; nosso objetivo é tornar o processamento explícito.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

ROOT = Path("/content/drive/MyDrive/Bioinformatica_Biologia_Molecular")
RAW = ROOT / "03_sra_fastq"
BASE = ROOT / "04_qc_trimming"
BASE.mkdir(parents=True, exist_ok=True)

SRR = "SRR15736591"
R1 = RAW / f"{SRR}_1.fastq.gz"
R2 = RAW / f"{SRR}_2.fastq.gz"

assert R1.exists() and R2.exists(), "Execute primeiro o notebook 03_FASTA_FASTQ_SRA.ipynb"
print(R1, R2, sep="\n")

## 1. Instalar FastQC, MultiQC e Trimmomatic

In [ ]:
!apt-get -qq update
!apt-get -qq install -y fastqc trimmomatic
!pip -q install multiqc
!fastqc --version
!trimmomatic -version
!multiqc --version

## 2. FastQC — reads brutos

In [ ]:
raw_qc = BASE / "fastqc_raw"
raw_qc.mkdir(exist_ok=True)
!fastqc -t 2 -o "$raw_qc" "$R1" "$R2"
!ls -lh "$raw_qc"

## 3. MultiQC — reads brutos

In [ ]:
multi_raw = BASE / "multiqc_raw"
multi_raw.mkdir(exist_ok=True)
!multiqc "$raw_qc" -o "$multi_raw" -f

## 4. Localizar arquivo de adaptadores do Trimmomatic

In [ ]:
import subprocess

cmd = "find /usr/share -name 'TruSeq3-PE.fa' 2>/dev/null | head -1"
adapter = subprocess.check_output(cmd, shell=True, text=True).strip()
print("Adapters:", adapter)

if not adapter:
    raise FileNotFoundError("Arquivo TruSeq3-PE.fa não encontrado.")

## 5. Trimmomatic paired-end

Parâmetros didáticos:
- ILLUMINACLIP: remoção de adaptadores;
- LEADING:5;
- TRAILING:15;
- SLIDINGWINDOW:4:15;
- MINLEN:40.

In [ ]:
R1P = BASE / f"{SRR}_R1_paired.fastq.gz"
R1U = BASE / f"{SRR}_R1_unpaired.fastq.gz"
R2P = BASE / f"{SRR}_R2_paired.fastq.gz"
R2U = BASE / f"{SRR}_R2_unpaired.fastq.gz"

!trimmomatic PE -threads 2 -phred33   "$R1" "$R2"   "$R1P" "$R1U"   "$R2P" "$R2U"   ILLUMINACLIP:"$adapter":2:30:10:2:keepBothReads   LEADING:5 TRAILING:15 SLIDINGWINDOW:4:15 MINLEN:40

## 6. Contar reads paired retidos

In [ ]:
import gzip

def nreads(path):
    with gzip.open(path, "rt") as f:
        return sum(1 for _ in f)//4

print("R1 bruto:", nreads(R1))
print("R1 paired após trimming:", nreads(R1P))
print("R2 bruto:", nreads(R2))
print("R2 paired após trimming:", nreads(R2P))

## 7. FastQC e MultiQC após trimming

In [ ]:
trim_qc = BASE / "fastqc_trimmed"
trim_qc.mkdir(exist_ok=True)
!fastqc -t 2 -o "$trim_qc" "$R1P" "$R2P"

multi_trim = BASE / "multiqc_trimmed"
multi_trim.mkdir(exist_ok=True)
!multiqc "$trim_qc" -o "$multi_trim" -f

## 8. Discussão

Compare os relatórios antes e depois.

Perguntas:
- a distribuição de qualidade mudou?
- houve indicação de adaptadores?
- quantos pares foram mantidos?
- faria sentido aplicar filtros ainda mais severos?